# Avance 2 — Asistente multimodal para la identificación de servicios REST

Procesamiento de lenguaje natural (TF-IDF + SVM) y clasificación de imágenes (ResNet50 con transferencia de aprendizaje) integrados al chatbot del Avance 1.

**Universidad Tecmilenio** · Desarrollo de aplicaciones con inteligencia artificial  
**Alumno:** Alejandro López Arteaga · **Matrícula:** 07302961

Funciona en VS Code (extensión Jupyter, abierto desde la carpeta del repositorio) y en Google Colab (clona el repositorio).

## 1. Entorno

In [ ]:
import sys, os
print(sys.version)
EN_COLAB = 'google.colab' in sys.modules
# En Colab se clona el repositorio (rama develop) una sola vez
if EN_COLAB and not os.path.exists('asistente'):
    if not os.path.exists('chatbot-tm'):
        !git clone -q -b develop https://github.com/lopartale/chatbot-tm.git
    os.chdir('chatbot-tm')
if EN_COLAB:
    %pip install -q -r requirements.txt
    !playwright install --with-deps chromium > /dev/null
import nltk, sklearn, keras
print('NLTK', nltk.__version__, '| scikit-learn', sklearn.__version__, '| Keras', keras.__version__, f'({keras.backend.backend()})')

## 2. Conjunto de datos textual
634 frases etiquetadas en las 9 intenciones del diseño conversacional: 508 de entrenamiento y 126 de prueba (escritas antes de entrenar).

In [ ]:
import pandas as pd
from lenguaje.representacion_texto import INTENCIONES, cargar_conjunto_intenciones
entrenamiento, prueba = cargar_conjunto_intenciones()
frases = pd.concat([entrenamiento, prueba], ignore_index=True)
resumen = pd.crosstab(frases.intencion, frases.conjunto).reindex(INTENCIONES)
resumen['dificiles'] = pd.crosstab(frases.intencion, frases.tipo).reindex(INTENCIONES)['dificil']
display(resumen)
frases.sample(8, random_state=1)

## 3. Preprocesamiento de texto
Normalización → tokenización (NLTK) → eliminación de palabras vacías (NLTK, conservando las que distinguen intenciones) → stemming (Snowball).

In [ ]:
from lenguaje.preprocesamiento import mostrar_pasos
from evaluacion.evaluar_intenciones import vocabulario_por_paso
for ejemplo in ['¿Qué hago si no existe el servicio que necesito?', 'Detalle de SRV-012', 'ayudaaa, q puedes hacer']:
    for paso, resultado in mostrar_pasos(ejemplo).items():
        print(f'{paso:20}: {resultado}')
    print()
pd.DataFrame(vocabulario_por_paso(list(entrenamiento.texto)), columns=['paso', 'términos distintos', 'términos totales'])

## 4. Representación TF-IDF

In [ ]:
from lenguaje.representacion_texto import crear_vectorizador
from evaluacion.evaluar_intenciones import terminos_representativos
vectorizador = crear_vectorizador()
matriz = vectorizador.fit_transform(entrenamiento.texto)
print('Matriz de entrenamiento:', matriz.shape, f'- densidad {100 * matriz.nnz / (matriz.shape[0] * matriz.shape[1]):.2f} %')
for intencion, terminos in terminos_representativos(vectorizador, matriz, entrenamiento.intencion, cantidad=5).items():
    print(f'{intencion:24}', ', '.join(termino for termino, _ in terminos))

## 5. Clasificador de intenciones (SVM) y evaluación

In [ ]:
from evaluacion import evaluar_intenciones
evaluar_intenciones.main()   # entrena el SVM, lo evalúa, lo compara con otras configuraciones y guarda evidencias/

In [ ]:
from IPython.display import Image as Imagen
Imagen('evidencias/matriz_confusion_intenciones.png', width=650)

## 6. Conjunto de imágenes
1,360 pantallas sintéticas de una banca digital en 8 tipos (cada uno ligado a un dominio del RAML). Estilos de diseño final y maqueta para entrenar/validar/probar; bocetos como estilo no visto.

In [ ]:
from vision.generar_pantallas import CARPETA_IMAGENES, crear_hoja_de_muestra, generar_imagenes
if not (CARPETA_IMAGENES / 'imagenes.csv').exists():
    generar_imagenes()     # ~1-2 minutos
crear_hoja_de_muestra()
imagenes = pd.read_csv(CARPETA_IMAGENES / 'imagenes.csv')
display(pd.crosstab(imagenes.clase, imagenes.conjunto))
Imagen('evidencias/muestra_pantallas.png')

## 7. Clasificador visual: ResNet50 + transferencia de aprendizaje
ResNet50 preentrenada en ImageNet (`keras.applications`) congelada como extractor de 2048 características; se entrena una capa `Dense` softmax de Keras con las pantallas. La regularización y el umbral de confianza se eligen con el conjunto de validación.

In [ ]:
from vision.preparar_resnet50 import preparar_resnet50
preparar_resnet50()        # descarga los pesos de ResNet50 para Keras (~95 MB) y la deja congelada
from evaluacion import evaluar_pantallas
evaluar_pantallas.main()   # extrae características (~3 min en CPU), entrena la capa softmax y evalúa

In [ ]:
display(Imagen('evidencias/matriz_confusion_pantallas_prueba.png', width=600))
Imagen('evidencias/matriz_confusion_pantallas_estilo_nuevo.png', width=600)

## 8. Integración multimodal
Una imagen adjunta pasa por el clasificador visual; el tipo de pantalla se traduce en sus operaciones típicas y se cruzan con el inventario RAML. El texto lo clasifica el SVM y NLTK Chat redacta las respuestas de charla.

In [ ]:
from asistente.chatbot_servicios import AsistenteServiciosREST
bot = AsistenteServiciosREST()
turnos = [('Te comparto el mockup de la pantalla', None),
          ('Aquí está', CARPETA_IMAGENES / 'prueba/transferencia/transferencia_003.jpg'),
          ('dame el contrato del tercero', None),
          ('¿Y este boceto?', CARPETA_IMAGENES / 'estilo_nuevo/seguros/seguros_000.jpg')]
for mensaje, imagen in turnos:
    print('Tú:', mensaje, f'[imagen: {imagen.name}]' if imagen else '')
    print('Bot:', bot.responder(mensaje, imagen=imagen), '\n')

In [ ]:
from evaluacion import pruebas_multimodales
resultados = pruebas_multimodales.ejecutar_conversaciones()
turnos = [turno for conversacion in resultados.values() for turno in conversacion]
print(sum(turno['correcto'] for turno in turnos), 'de', len(turnos), 'turnos correctos')

## 9. Pruebas automáticas

In [ ]:
!{sys.executable} -m pytest -q

## 10. Modo interactivo
Escribe `salir` para terminar. Para analizar una captura: `/imagen ruta/de/la/imagen.png`.

In [ ]:
from asistente.chatbot_servicios import main
main()